# NeoOLAF native EventStoryLine layer ablation — one document v1.3

This v1.3 notebook keeps the same **OWL-Time** ontology used in RAGTree and the controlled labels `PRECONDITION`/`FALLING_ACTION`. Layer 1 now uses parallel sentence-level event inventories, two whole-document coverage reviews, and source-batched closed-inventory PLOT_LINK generation. Gold remains unavailable until after Layer 12.


In [1]:
from __future__ import annotations

import os
import sys
from getpass import getpass
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/neoolaf").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the NeoOLAF repository.")


def first_existing_path(env_name: str, candidates: list[Path]) -> Path:
    raw = os.environ.get(env_name, "").strip().strip('"').strip("'")
    if raw:
        path = Path(raw).expanduser().resolve()
        if path.is_file():
            return path
        raise FileNotFoundError(f"{env_name} points to a missing file: {path}")
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Could not find {env_name}. Tried:\n"
        + "\n".join(str(Path(x).expanduser().resolve()) for x in candidates)
    )


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "examples/RAGTreeDatasets"
TOOLS_DIR = NOTEBOOK_DIR / "tools"
for path in [PROJECT_ROOT / "src", PROJECT_ROOT, TOOLS_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from eventstoryline_native_ablation_v1_3 import (
    RELATION_IDS,
    analyze_run,
    gold_event_index,
    indexed_token_table,
    load_layer_states,
    project_event_label,
    read_json,
    read_jsonl,
    run_native_pipeline,
    seed_ontology_summary,
)

print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = /home/galencarmedeiro/git/postdoc/NeoOLAF


/home/galencarmedeiro/git/postdoc/NeoOLAF/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
/home/galencarmedeiro/git/postdoc/NeoOLAF/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuration

In [2]:
INPUT_JSONL = NOTEBOOK_DIR / "data/eventstoryline_one_input_v1.jsonl"
GOLD_JSONL = NOTEBOOK_DIR / "data/eventstoryline_one_gold_v1.jsonl"
SMOKE5_INPUT_JSONL = NOTEBOOK_DIR / "data/eventstoryline_smoke5_input_v1.jsonl"
SMOKE5_GOLD_JSONL = NOTEBOOK_DIR / "data/eventstoryline_smoke5_gold_v1.jsonl"

# Same seed ontology used by the RAGTree EventStoryLine experiments.
# Override explicitly with EVENTSTORYLINE_ONTOLOGY_PATH when needed.
ONTOLOGY_PATH = first_existing_path(
    "EVENTSTORYLINE_ONTOLOGY_PATH",
    [
        PROJECT_ROOT.parent / "ragtree/data/ontology/OWLTime/time.ttl",
        PROJECT_ROOT.parent / "RAGTree/data/ontology/OWLTime/time.ttl",
        PROJECT_ROOT / "../ragtree/data/ontology/OWLTime/time.ttl",
        PROJECT_ROOT / "../RAGTree/data/ontology/OWLTime/time.ttl",
    ],
)

# Controlled normalized benchmark relation schema. This is not the seed ontology.
RELATION_CATALOG = NOTEBOOK_DIR / "ontology/eventstoryline_relation_catalog.json"
RELATION_ALIASES = NOTEBOOK_DIR / "ontology/eventstoryline_relation_aliases.json"
PROFILE_PATH = NOTEBOOK_DIR / "configs/eventstoryline_profile_native_ablation_v1_3.json"
GUIDANCE_PATH = NOTEBOOK_DIR / "configs/guidance_eventstoryline_native_ablation_v1_3.json"
TASK_GUIDANCE_PATH = NOTEBOOK_DIR / "configs/eventstoryline_task_guidance_v1_3.json"

RUNS_ROOT = NOTEBOOK_DIR / "runs/eventstoryline_native_layer_ablation"
RUN_DIR = RUNS_ROOT / "document_1_10ecbplus_v1_3_owltime_recall"

OPENROUTER_HOST = "https://openrouter.ai/api/v1"
MODEL_NAME = "openai/gpt-oss-20b"
API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")

# Relation-instance decisions are independent and run concurrently in Layer 2.
WORKERS = 16
REASONING_EFFORT = "minimal"
RUN_PIPELINE = True
CLEAN_RUN_DIR = True

print("Input:", INPUT_JSONL)
print("Gold:", GOLD_JSONL)
print("OWL-Time seed ontology:", ONTOLOGY_PATH)
print("Controlled relation catalog:", RELATION_CATALOG)
print("Profile:", PROFILE_PATH)
print("Guidance:", GUIDANCE_PATH)
print("Task guidance:", TASK_GUIDANCE_PATH)
print("Run dir:", RUN_DIR)
print("Model:", MODEL_NAME)
print("API key available:", bool(API_KEY))


Input: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/data/eventstoryline_one_input_v1.jsonl
Gold: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/data/eventstoryline_one_gold_v1.jsonl
OWL-Time seed ontology: /home/galencarmedeiro/git/postdoc/ragtree/data/ontology/OWLTime/time.ttl
Controlled relation catalog: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/ontology/eventstoryline_relation_catalog.json
Profile: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/configs/eventstoryline_profile_native_ablation_v1_3.json
Guidance: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/configs/guidance_eventstoryline_native_ablation_v1_3.json
Task guidance: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/configs/eventstoryline_task_guidance_v1_3.json
Run dir: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/eventstoryline_native_layer_ablation/document_1_10ecbplus_v

## 2. Preflight: anti-leakage, OWL-Time seed ontology, five-document files and event identity


In [3]:
required = [
    INPUT_JSONL, GOLD_JSONL, SMOKE5_INPUT_JSONL, SMOKE5_GOLD_JSONL,
    ONTOLOGY_PATH, RELATION_CATALOG, RELATION_ALIASES,
    PROFILE_PATH, GUIDANCE_PATH, TASK_GUIDANCE_PATH,
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

input_rows = read_jsonl(INPUT_JSONL)
gold_rows = read_jsonl(GOLD_JSONL)
smoke_input_rows = read_jsonl(SMOKE5_INPUT_JSONL)
smoke_gold_rows = read_jsonl(SMOKE5_GOLD_JSONL)
assert len(input_rows) == 1 and len(gold_rows) == 1
assert len(smoke_input_rows) == 5 and len(smoke_gold_rows) == 5
assert "entities" not in input_rows[0] and "relations" not in input_rows[0]
assert all("entities" not in row and "relations" not in row for row in smoke_input_rows)
assert input_rows[0]["document_id"] == gold_rows[0]["document_id"]
assert [x["document_id"] for x in smoke_input_rows] == [x["document_id"] for x in smoke_gold_rows]

profile = read_json(PROFILE_PATH)
task = read_json(TASK_GUIDANCE_PATH)
catalog = read_json(RELATION_CATALOG)
gold = gold_rows[0]
seed_summary = seed_ontology_summary(ONTOLOGY_PATH)

print("Document:", input_rows[0]["document_id"], "-", input_rows[0]["title"])
print("Source characters:", len(input_rows[0]["text"]))
print("Sentences:", len(input_rows[0]["sentences"]))
print("Source tokens:", sum(len(x) for x in input_rows[0]["tokens"]))
print("Gold events (not exposed to pipeline):", len(gold["entities"]))
print("Gold evaluated relations:", sum(len(v) for k, v in gold["relations"].items() if k in RELATION_IDS))
print("Ignored null pairs:", len(gold["relations"].get("null", [])))
print("OWL-Time classes loaded:", seed_summary["class_count"])
print("OWL-Time properties loaded:", seed_summary["property_count"])
print("Controlled task relations:", catalog["property_count"])
print("Relation IDs:", task["allowed_relation_ids"])
print("Layer 1 sentence workers:", profile["layers"]["layer01_linguistic_expression_extraction"]["sentence_workers"])
print("Layer 1 coverage reviews:", profile["layers"]["layer01_linguistic_expression_extraction"]["coverage_review_passes"])
print("Layer 1 relation workers:", profile["layers"]["layer01_linguistic_expression_extraction"]["relation_workers"])
print("Layer 1 source batch size:", profile["layers"]["layer01_linguistic_expression_extraction"]["relation_source_batch_size"])
print("Layer 2 workers:", profile["layers"]["layer02_candidate_enrichment"]["max_concurrency"])
print("Mention-free relation schemas injected:", len(profile["relations"]["allowed"]))
print("Five-document input IDs:", [row["document_id"] for row in smoke_input_rows])

assert seed_summary["class_count"] > 0 or seed_summary["property_count"] > 0
assert catalog["property_count"] == 2
assert set(task["allowed_relation_ids"]) == set(RELATION_IDS)
assert profile["relations"]["allowed"] == []
assert profile["anti_cheating"]["direct_eventstoryline_extraction"] is False
assert profile["anti_cheating"]["source_event_anchoring"] is False
assert profile["anti_cheating"]["gold_pair_hints"] is False
assert profile["anti_cheating"]["post_run_relation_invention"] is False
assert profile["benchmark_projection"]["gold_available_to_pipeline"] is False
assert profile["benchmark_projection"]["same_seed_ontology_as_ragtree"] is True

display(Markdown("### Indexed source table used by Layer 1"))
print(indexed_token_table(input_rows[0]["sentences"], input_rows[0]["tokens"]))

assert profile["anti_cheating"]["gold_event_lexicon"] is False
assert profile["anti_cheating"]["gold_event_count"] is False
assert profile["anti_cheating"]["gold_relation_count"] is False


Document: EventStoryLine - 1_10ecbplus - 1_10ecbplus
Source characters: 745
Sentences: 6
Source tokens: 147
Gold events (not exposed to pipeline): 15
Gold evaluated relations: 20
Ignored null pairs: 1
OWL-Time classes loaded: 23
OWL-Time properties loaded: 62
Controlled task relations: 2
Relation IDs: ['PRECONDITION', 'FALLING_ACTION']
Layer 1 sentence workers: 8
Layer 1 coverage reviews: 2
Layer 1 relation workers: 8
Layer 1 source batch size: 4
Layer 2 workers: 16
Mention-free relation schemas injected: 0
Five-document input IDs: ['EventStoryLine - 1_10ecbplus', 'EventStoryLine - 1_11ecbplus', 'EventStoryLine - 1_12ecbplus', 'EventStoryLine - 1_13ecbplus', 'EventStoryLine - 1_14ecbplus']


### Indexed source table used by Layer 1

[S0] http : / / articles . latimes . com / 2013 / may / 03 / local / la - me - 0504 - lohan - rehab - 20130504
TOKENS 0=http 1=: 2=/ 3=/ 4=articles 5=. 6=latimes 7=. 8=com 9=/ 10=2013 11=/ 12=may 13=/ 14=03 15=/ 16=local 17=/ 18=la 19=- 20=me 21=- 22=0504 23=- 24=lohan 25=- 26=rehab 27=- 28=20130504

[S1] Lindsay Lohan checks into Betty Ford Center
TOKENS 0=Lindsay 1=Lohan 2=checks 3=into 4=Betty 5=Ford 6=Center

[S2] May 03 , 2013
TOKENS 0=May 1=03 2=, 3=2013

[S3] After skipping out on entering a Newport Beach rehabilitation facility and facing the prospect of arrest for violating her probation , Lindsay Lohan has checked into the Betty Ford Center to begin a 90 - day court - mandated stay in her reckless driving conviction .
TOKENS 0=After 1=skipping 2=out 3=on 4=entering 5=a 6=Newport 7=Beach 8=rehabilitation 9=facility 10=and 11=facing 12=the 13=prospect 14=of 15=arrest 16=for 17=violating 18=her 19=probation 20=, 21=Lindsay 22=Lohan 23=has 24=checked 25=into 26=the 27=Betty 28=Fo

## 3. Run the full native Layer 0--12 pipeline

In [4]:
if RUN_PIPELINE:
    if not API_KEY:
        API_KEY = getpass("OpenRouter API key: ").strip().strip('"').strip("'")
    if not API_KEY:
        raise RuntimeError("No OpenRouter API key was provided.")

    final_state = run_native_pipeline(
        project_root=PROJECT_ROOT,
        input_jsonl=INPUT_JSONL,
        ontology_path=ONTOLOGY_PATH,
        profile_path=PROFILE_PATH,
        guidance_path=GUIDANCE_PATH,
        task_guidance_path=TASK_GUIDANCE_PATH,
        relation_catalog_path=RELATION_CATALOG,
        relation_aliases_path=RELATION_ALIASES,
        run_dir=RUN_DIR,
        model_name=MODEL_NAME,
        api_key=API_KEY,
        host=OPENROUTER_HOST,
        workers=WORKERS,
        reasoning_effort=REASONING_EFFORT,
        verbose=True,
        clean_run_dir=CLEAN_RUN_DIR,
    )
    print("Full native run completed.")
else:
    print("RUN_PIPELINE=False: reusing", RUN_DIR)

[NeoOLAF] Run directory: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/eventstoryline_native_layer_ablation/document_1_10ecbplus_v1_3_owltime_recall
[NeoOLAF] from_layer=0, to_layer=12, skip_layers=None
[NeoOLAF] Pipeline has 13 layers
[NeoOLAF] Selected layers: ['layer00_preprocessing', 'layer01_linguistic_expression_extraction', 'layer02_candidate_enrichment', 'layer03_candidate_typing_resolution', 'layer04_candidate_relation_extraction', 'layer05_candidate_triple_generation', 'layer06_concept_relation_induction', 'layer07_hierarchisation', 'layer08_axiom_schemata_extraction', 'layer09_general_axiom_extraction', 'layer10_validation_reasoning', 'layer11_inference_completion', 'layer12_serialization']
[NeoOLAF] Layer 0/12: layer00_preprocessing

[NeoOLAF] Starting layer: layer00_preprocessing
[NeoOLAF] Finished layer: layer00_preprocessing in 0.01s
[NeoOLAF] Layer 1/12: layer01_linguistic_expression_extraction

[NeoOLAF] Starting layer: layer01_linguistic_expr

[NeoOLAF] Finished layer: layer03_candidate_typing_resolution in 0.01s
[NeoOLAF] Layer 4/12: layer04_candidate_relation_extraction

[NeoOLAF] Starting layer: layer04_candidate_relation_extraction
[NeoOLAF][Layer 4] strategy=structured_exact_then_native_parallel_fallback; parallel_workers=8; attempts=1
[NeoOLAF] Finished layer: layer04_candidate_relation_extraction in 0.01s
[NeoOLAF] Layer 5/12: layer05_candidate_triple_generation

[NeoOLAF] Starting layer: layer05_candidate_triple_generation


[NeoOLAF] Finished layer: layer05_candidate_triple_generation in 0.00s
[NeoOLAF] Layer 6/12: layer06_concept_relation_induction

[NeoOLAF] Starting layer: layer06_concept_relation_induction
[NeoOLAF][Layer 6] deterministic ontology-aware concept induction for 18 node candidates; no LLM calls.
[NeoOLAF][Layer 6] deterministic ontology-aware relation induction for 26 relation candidates; no LLM calls.
[NeoOLAF] Finished layer: layer06_concept_relation_induction in 0.00s
[NeoOLAF] Layer 7/12: layer07_hierarchisation

[NeoOLAF] Starting layer: layer07_hierarchisation
[NeoOLAF] Finished layer: layer07_hierarchisation in 0.00s
[NeoOLAF] Layer 8/12: layer08_axiom_schemata_extraction

[NeoOLAF] Starting layer: layer08_axiom_schemata_extraction
[NeoOLAF][Layer 8] strategy=ontology_aware_axiom_schema_generation
[NeoOLAF] Finished layer: layer08_axiom_schemata_extraction in 0.00s
[NeoOLAF] Layer 9/12: layer09_general_axiom_extraction

[NeoOLAF] Starting layer: layer09_general_axiom_extraction
[Ne

[NeoOLAF] Finished layer: layer10_validation_reasoning in 0.00s
[NeoOLAF] Layer 11/12: layer11_inference_completion

[NeoOLAF] Starting layer: layer11_inference_completion
[NeoOLAF][Layer 11] strategy=ontology_aware_semantic_completion
[NeoOLAF][Layer 11] deterministic completion; max_concurrency=16; no LLM calls.
[NeoOLAF] Finished layer: layer11_inference_completion in 0.00s
[NeoOLAF] Layer 12/12: layer12_serialization

[NeoOLAF] Starting layer: layer12_serialization
[NeoOLAF] Exports written to: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/eventstoryline_native_layer_ablation/document_1_10ecbplus_v1_3_owltime_recall/exports
[NeoOLAF] Finished layer: layer12_serialization in 0.04s
[NeoOLAF] Pipeline finished in 203.56s
[NeoOLAF] Saved checkpoint: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/eventstoryline_native_layer_ablation/document_1_10ecbplus_v1_3_owltime_recall/checkpoints/after_selected_pipeline.pkl.gz
[NeoOLAF] Total run t

### Runtime evidence saved

- `run_manifest.json` records OWL-Time and the v1.3 operational fingerprint.
- `run_logs/layer01_call_audit.json` records sentence calls, coverage reviews, and source batches.
- `run_logs/layer01_event_inventory.json` records every accepted/rejected span proposal.
- `run_logs/layer01_relation_generation.json` records closed-inventory pair proposals.
- `run_logs/layer01_event_relation_instances.json` records materialized expressions.
- Layer 2/4 decisions, ontology retrieval, API responses, and all Layer 0--12 states remain saved.


## 4. Strict event and relation evaluation

In [5]:
summary = analyze_run(
    run_dir=RUN_DIR,
    gold_jsonl=GOLD_JSONL,
    catalog_path=RELATION_CATALOG,
    aliases_path=RELATION_ALIASES,
)

display(pd.DataFrame(summary["layer_summary"]))

print("Strict relation evaluation; null relations excluded")
display(pd.DataFrame([summary["strict_relation_evaluation"]]))

print("Event mention inventory evaluation")
display(pd.DataFrame([summary["event_entity_evaluation"]]))

print("Relation-endpoint event inventory evaluation")
display(pd.DataFrame([summary["relation_endpoint_evaluation"]]))

print("Per-relation metrics")
display(pd.DataFrame(summary["per_relation_metrics"]))

print("Cumulative evaluation")
display(pd.DataFrame(summary["cumulative_evaluation"]))

print("First-failure counts")
print(summary["failure_counts"])

,layer,layer_name,linguistic_expressions,enriched_expressions,entity_candidates,relation_candidates,attribute_candidates,event_candidates,candidate_relation_assertions,candidate_triples,concept_candidates,ontology_relation_candidates,concept_hierarchy_links,relation_hierarchy_links,axiom_schema_candidates,general_axiom_candidates,completion_candidates,validation_issues,reasoning_inferred_triples
0,0,layer00_preprocessing,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1,layer01_linguistic_expression_extraction,44,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,2,layer02_candidate_enrichment,44,44,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,3,layer03_candidate_typing_resolution,44,44,0,26,0,18,0,0,0,0,0,0,0,0,0,0,0
4,4,layer04_candidate_relation_extraction,44,44,0,26,0,18,26,0,0,0,0,0,0,0,0,0,0
5,5,layer05_candidate_triple_generation,44,44,0,26,0,18,26,26,0,0,0,0,0,0,0,0,0
6,6,layer06_concept_relation_induction,44,44,0,26,0,18,26,26,0,3,0,0,0,0,0,0,0
7,7,layer07_hierarchisation,44,44,0,26,0,18,26,26,0,3,0,3,0,0,0,0,0
8,8,layer08_axiom_schemata_extraction,44,44,0,26,0,18,26,26,0,3,0,3,7,0,0,0,0
9,9,layer09_general_axiom_extraction,44,44,0,26,0,18,26,26,0,3,0,3,7,10,0,0,0


Strict relation evaluation; null relations excluded


,predicted,gold,true_positive,false_positive,false_negative,precision,recall,f1
0,15,20,2,13,18,0.133333,0.1,0.114286


Event mention inventory evaluation


,predicted,gold,true_positive,false_positive,false_negative,precision,recall,f1
0,12,15,12,0,3,1.0,0.8,0.888889


...

Per-relation metrics


,relation_id,predicted,gold,true_positive,false_positive,false_negative,precision,recall,f1
0,PRECONDITION,13,7,2,11,5,0.153846,0.285714,0.2
1,FALLING_ACTION,2,13,0,2,13,0.000000,0.000000,0.0


Cumulative evaluation


,layer,layer_name,relation_predicted,relation_gold,relation_true_positive,relation_false_positive,relation_false_negative,relation_precision,relation_recall,relation_f1,event_predicted,event_gold,event_true_positive,event_false_positive,event_false_negative,event_precision,event_recall,event_f1
0,0,layer00_preprocessing,0,20,0,0,20,0.000000,0.0,0.000000,0,15,0,0,15,0.0,0.0,0.000000
1,1,layer01_linguistic_expression_extraction,0,20,0,0,20,0.000000,0.0,0.000000,12,15,12,0,3,1.0,0.8,0.888889
2,2,layer02_candidate_enrichment,0,20,0,0,20,0.000000,0.0,0.000000,12,15,12,0,3,1.0,0.8,0.888889
3,3,layer03_candidate_typing_resolution,0,20,0,0,20,0.000000,0.0,0.000000,12,15,12,0,3,1.0,0.8,0.888889
4,4,layer04_candidate_relation_extraction,15,20,2,13,18,0.133333,0.1,0.114286,12,15,12,0,3,1.0,0.8,0.888889
5,5,layer05_candidate_triple_generation,15,20,2,13,18,0.133333,0.1,0.114286,12,15,12,0,3,1.0,0.8,0.888889
6,6,layer06_concept_relation_induction,15,20,2,13,18,0.133333,0.1,0.114286,12,15,12,0,3,1.0,0.8,0.888889
7,7,layer07_hierarchisation,15,20,2,13,18,0.133333,0.1,0.114286,12,15,12,0,3,1.0,0.8,0.888889
8,8,layer08_axiom_schemata_extraction,15,20,2,13,18,0.133333,0.1,0.114286,12,15,12,0,3,1.0,0.8,0.888889
9,9,layer09_general_axiom_extraction,15,20,2,13,18,0.133333,0.1,0.114286,12,15,12,0,3,1.0,0.8,0.888889


First-failure counts
{'layer01_relation_instance_missing': 11, 'survived_to_layer05': 2, 'layer01_target_event_missing': 4, 'layer01_source_event_missing': 3}


## 5. Layer 1 validated event inventory and closed-inventory relation instances

In [6]:
layer1_rows = read_json(RUN_DIR / "run_logs/layer01_event_relation_instances.json")
layer1_df = pd.DataFrame(layer1_rows)
display(layer1_df)

accepted = layer1_df[layer1_df["status"] == "accepted"] if not layer1_df.empty else layer1_df
if not accepted.empty:
    print("Accepted event mentions:", int((accepted["label"] == "event_mention").sum()))
    print("Accepted relation instances:", int((accepted["label"] == "relation_instance").sum()))
    print("Rejected rows:", int((layer1_df["status"] == "rejected").sum()))

,phase,batch_index,assigned_source_ids,source_event_id,target_event_id,relation_cue,justification,status,reason,label,expr_id,text,relation_instance
0,relation_source_batch,2.0,"[E0008, E0009, E0010, E0011]",E0009,E0008,precondition,The 90‑day stay is a consequence of the reckle...,rejected,missing_or_canonical_relation_cue,relation_instance,NaN,NaN,NaN
1,relation_source_batch,2.0,"[E0008, E0009, E0010, E0011]",E0010,E0009,precondition,The probation violation led to the reckless dr...,rejected,missing_or_canonical_relation_cue,relation_instance,NaN,NaN,NaN
2,relation_source_batch,2.0,"[E0008, E0009, E0010, E0011]",E0011,E0009,precondition,The prior drunk driving case (driving) resulte...,rejected,missing_or_canonical_relation_cue,relation_instance,NaN,NaN,NaN
3,materialization,NaN,NaN,NaN,NaN,NaN,Explicit event.,accepted,NaN,event_mention,expr_00000,S1[2:4]::checks into,None
4,materialization,NaN,NaN,NaN,NaN,NaN,Explicit verb.,accepted,NaN,event_mention,expr_00001,S3[1:2]::skipping,None
5,materialization,NaN,NaN,NaN,NaN,NaN,Explicit verb.,accepted,NaN,event_mention,expr_00002,S3[4:5]::entering,None
6,materialization,NaN,NaN,NaN,NaN,NaN,Explicit verb.,accepted,NaN,event_mention,expr_00003,S3[11:12]::facing,None
7,materialization,NaN,NaN,NaN,NaN,NaN,Explicit eventive noun.,accepted,NaN,event_mention,expr_00004,S3[15:16]::arrest,None
8,materialization,NaN,NaN,NaN,NaN,NaN,Explicit verb.,accepted,NaN,event_mention,expr_00005,S3[17:18]::violating,None
9,materialization,NaN,NaN,NaN,NaN,NaN,Explicit verb.,accepted,NaN,event_mention,expr_00006,S3[24:26]::checked into,None


Accepted event mentions: 18
Accepted relation instances: 26
Rejected rows: 3


### Layer 1A span validation/repair and Layer 1B closed-inventory audit


In [7]:
inventory_audit = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer01_event_inventory.json"))
relation_generation = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer01_relation_generation.json"))
call_audit = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer01_call_audit.json"))

print("Layer 1A event inventory audit")
display(inventory_audit)
if not inventory_audit.empty:
    print("Accepted validated events:", int((inventory_audit["status"] == "accepted").sum()))
    print("Rejected event proposals:", int((inventory_audit["status"] == "rejected").sum()))
    if "repaired" in inventory_audit:
        print("Source-token span repairs:", int(inventory_audit["repaired"].fillna(False).sum()))

print("Layer 1B relation generation audit")
display(relation_generation)
if not relation_generation.empty:
    print("Accepted closed-inventory pairs:", int((relation_generation["status"] == "accepted").sum()))
    print("Rejected pair proposals:", int((relation_generation["status"] == "rejected").sum()))

print("Layer 1 sentence/review/source-batch call audit")
display(call_audit)
if not call_audit.empty:
    display(call_audit.groupby(["phase", "status"], dropna=False).size().reset_index(name="calls"))


Layer 1A event inventory audit


,raw_item,proposed_event_key,proposed_sentence_id,proposed_token_start,proposed_token_end,proposed_trigger,repair_steps,status,reason,canonical_event_key,sentence_id,token_start,token_end,repaired_trigger,repaired,phase,justification
0,"{'sentence_id': 1, 'token_start': 2, 'token_en...",None,1,2,4,checks into,[],accepted,validated_source_token_span,S1[2:4]::checks into,1,2,4,checks into,False,sentence_inventory,Explicit event.
1,"{'sentence_id': 3, 'token_start': 1, 'token_en...",None,3,1,2,skipping,[],accepted,validated_source_token_span,S3[1:2]::skipping,3,1,2,skipping,False,sentence_inventory,Explicit verb.
2,"{'sentence_id': 3, 'token_start': 4, 'token_en...",None,3,4,5,entering,[],accepted,validated_source_token_span,S3[4:5]::entering,3,4,5,entering,False,sentence_inventory,Explicit verb.
3,"{'sentence_id': 3, 'token_start': 11, 'token_e...",None,3,11,12,facing,[],accepted,validated_source_token_span,S3[11:12]::facing,3,11,12,facing,False,sentence_inventory,Explicit verb.
4,"{'sentence_id': 3, 'token_start': 15, 'token_e...",None,3,15,16,arrest,[],accepted,validated_source_token_span,S3[15:16]::arrest,3,15,16,arrest,False,sentence_inventory,Explicit eventive noun.
5,"{'sentence_id': 3, 'token_start': 17, 'token_e...",None,3,17,18,violating,[],accepted,validated_source_token_span,S3[17:18]::violating,3,17,18,violating,False,sentence_inventory,Explicit verb.
6,"{'sentence_id': 3, 'token_start': 24, 'token_e...",None,3,24,25,checked,[completed_phrasal_verb_particle],accepted,validated_source_token_span,S3[24:26]::checked into,3,24,26,checked into,True,sentence_inventory,Explicit verb.
7,"{'sentence_id': 3, 'token_start': 31, 'token_e...",None,3,31,32,begin,[],accepted,validated_source_token_span,S3[31:32]::begin,3,31,32,begin,False,sentence_inventory,Explicit verb.
8,"{'sentence_id': 3, 'token_start': 39, 'token_e...",None,3,39,40,stay,[],accepted,validated_source_token_span,S3[39:40]::stay,3,39,40,stay,False,sentence_inventory,Explicit eventive noun.
9,"{'sentence_id': 4, 'token_start': 14, 'token_e...",None,4,14,15,violation,[aligned_to_complete_trigger_text],accepted,validated_source_token_span,S4[15:16]::violation,4,15,16,violation,True,sentence_inventory,Explicit eventive noun.


Accepted validated events: 19
Rejected event proposals: 0
Source-token span repairs: 2
Layer 1B relation generation audit


,phase,batch_index,assigned_source_ids,source_event_id,target_event_id,relation_cue,justification,status,reason,relation_instance
0,relation_source_batch,0,"[E0000, E0001, E0002, E0003]",E0000,E0008,enables,Checking into the Betty Ford Center sets up th...,accepted,closed_inventory_source_batch_relation,S1[2:4]::checks into || enables || S3[39:40]::...
1,relation_source_batch,0,"[E0000, E0001, E0002, E0003]",E0001,E0000,enables,Skipping out on entering a rehab facility moti...,accepted,closed_inventory_source_batch_relation,S3[1:2]::skipping || enables || S1[2:4]::check...
2,relation_source_batch,0,"[E0000, E0001, E0002, E0003]",E0002,E0000,enables,The act of entering a Newport Beach facility (...,accepted,closed_inventory_source_batch_relation,S3[4:5]::entering || enables || S1[2:4]::check...
3,relation_source_batch,0,"[E0000, E0001, E0002, E0003]",E0003,E0000,motivates,Facing arrest prospect motivates the check‑in.,accepted,closed_inventory_source_batch_relation,S3[11:12]::facing || motivates || S1[2:4]::che...
4,relation_source_batch,1,"[E0004, E0005, E0006, E0007]",E0004,E0006,enables,Arrest is a prerequisite that leads to checkin...,accepted,closed_inventory_source_batch_relation,S3[15:16]::arrest || enables || S3[24:26]::che...
5,relation_source_batch,1,"[E0004, E0005, E0006, E0007]",E0005,E0004,enables,Violation of probation is a prerequisite that ...,accepted,closed_inventory_source_batch_relation,S3[17:18]::violating || enables || S3[15:16]::...
6,relation_source_batch,1,"[E0004, E0005, E0006, E0007]",E0005,E0006,enables,Violation of probation is a prerequisite that ...,accepted,closed_inventory_source_batch_relation,S3[17:18]::violating || enables || S3[24:26]::...
7,relation_source_batch,1,"[E0004, E0005, E0006, E0007]",E0006,E0007,enables,Checking into the center is a prerequisite tha...,accepted,closed_inventory_source_batch_relation,S3[24:26]::checked into || enables || S3[31:32...
8,relation_source_batch,2,"[E0008, E0009, E0010, E0011]",E0009,E0008,precondition,The 90‑day stay is a consequence of the reckle...,rejected,missing_or_canonical_relation_cue,NaN
9,relation_source_batch,2,"[E0008, E0009, E0010, E0011]",E0010,E0009,precondition,The probation violation led to the reckless dr...,rejected,missing_or_canonical_relation_cue,NaN


Accepted closed-inventory pairs: 26
Rejected pair proposals: 3
Layer 1 sentence/review/source-batch call audit


,phase,sentence_id,status,reason,proposals,new_validated_events,pass_index,inventory_size_after,batch_index,assigned_source_ids
0,sentence_inventory,0.0,skipped,url_sentence,NaN,NaN,NaN,NaN,NaN,NaN
1,sentence_inventory,2.0,skipped,date_or_publication_metadata,NaN,NaN,NaN,NaN,NaN,NaN
2,sentence_inventory,1.0,ok,NaN,1.0,1.0,NaN,NaN,NaN,NaN
3,sentence_inventory,3.0,ok,NaN,8.0,8.0,NaN,NaN,NaN,NaN
4,sentence_inventory,4.0,ok,NaN,2.0,2.0,NaN,NaN,NaN,NaN
5,sentence_inventory,5.0,ok,NaN,5.0,5.0,NaN,NaN,NaN,NaN
6,coverage_review,NaN,ok,NaN,1.0,0.0,1.0,16.0,NaN,NaN
7,coverage_review,NaN,ok,NaN,2.0,2.0,2.0,18.0,NaN,NaN
8,relation_source_batch,NaN,ok,NaN,4.0,NaN,NaN,NaN,1.0,"[E0004, E0005, E0006, E0007]"
9,relation_source_batch,NaN,ok,NaN,2.0,NaN,NaN,NaN,4.0,"[E0016, E0017]"


,phase,status,calls
0,coverage_review,ok,2
1,relation_source_batch,ok,5
2,sentence_inventory,ok,4
3,sentence_inventory,skipped,2


## 6. Parallel Layer 2 ontology decisions

In [8]:
layer2 = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_relation_decisions.json"))
prompt_audit = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_compact_prompt_audit.json"))

display(layer2)
print("Compact prompt audit")
display(prompt_audit)
if not prompt_audit.empty:
    print("Layer 2 calls:", len(prompt_audit))
    print("Mean Layer 2 user characters:", round(prompt_audit["user_chars"].mean(), 1))
    print("Maximum Layer 2 user characters:", int(prompt_audit["user_chars"].max()))
    print("Candidates per decision:", sorted(set(len(x) for x in prompt_audit["candidate_relation_ids"])))

,expr_id,relation_instance,source,predicate,target,found,selected_relation_id,canonical_relation,decision,ontology_hints
0,expr_00018,S1[2:4]::checks into || enables || S3[39:40]::...,S1[2:4]::checks into,enables,S3[39:40]::stay,True,PRECONDITION,PRECONDITION,The act of checking into the Betty Ford Center...,"[controlled_relation:PRECONDITION, promote_to_..."
1,expr_00019,S3[1:2]::skipping || enables || S1[2:4]::check...,S3[1:2]::skipping,enables,S1[2:4]::checks into,True,PRECONDITION,PRECONDITION,Skipping out on entering a rehab facility crea...,"[controlled_relation:PRECONDITION, promote_to_..."
2,expr_00020,S3[4:5]::entering || enables || S1[2:4]::check...,S3[4:5]::entering,enables,S1[2:4]::checks into,True,PRECONDITION,PRECONDITION,The act of entering a facility motivates and e...,"[controlled_relation:PRECONDITION, promote_to_..."
3,expr_00021,S3[11:12]::facing || motivates || S1[2:4]::che...,S3[11:12]::facing,motivates,S1[2:4]::checks into,True,PRECONDITION,PRECONDITION,The source event (facing arrest prospect) serv...,"[controlled_relation:PRECONDITION, promote_to_..."
4,expr_00022,S3[15:16]::arrest || enables || S3[24:26]::che...,S3[15:16]::arrest,enables,S3[24:26]::checked into,True,PRECONDITION,PRECONDITION,The arrest establishes a prerequisite conditio...,"[controlled_relation:PRECONDITION, promote_to_..."
5,expr_00023,S3[17:18]::violating || enables || S3[15:16]::...,S3[17:18]::violating,enables,S3[15:16]::arrest,True,PRECONDITION,PRECONDITION,Violation of probation is a prerequisite that ...,"[controlled_relation:PRECONDITION, promote_to_..."
6,expr_00024,S3[17:18]::violating || enables || S3[24:26]::...,S3[17:18]::violating,enables,S3[24:26]::checked into,True,PRECONDITION,PRECONDITION,Violation of probation is a prerequisite that ...,"[controlled_relation:PRECONDITION, promote_to_..."
7,expr_00025,S3[24:26]::checked into || enables || S3[31:32...,S3[24:26]::checked into,enables,S3[31:32]::begin,True,PRECONDITION,PRECONDITION,The act of checking into the center establishe...,"[controlled_relation:PRECONDITION, promote_to_..."
8,expr_00026,S5[6:7]::comes || comes after || S3[24:26]::ch...,S5[6:7]::comes,comes after,S3[24:26]::checked into,True,FALLING_ACTION,FALLING_ACTION,The source event 'comes' occurs after the targ...,"[controlled_relation:FALLING_ACTION, promote_t..."
9,expr_00027,S5[6:7]::comes || comes after || S3[31:32]::begin,S5[6:7]::comes,comes after,S3[31:32]::begin,True,PRECONDITION,PRECONDITION,The arrival event (comes) is the prerequisite ...,"[controlled_relation:PRECONDITION, promote_to_..."


Compact prompt audit


,expr_id,relation_instance,candidate_relation_ids,system_chars,user_chars
0,expr_00018,S1[2:4]::checks into || enables || S3[39:40]::...,"[PRECONDITION, FALLING_ACTION]",955,5007
1,expr_00019,S3[1:2]::skipping || enables || S1[2:4]::check...,"[PRECONDITION, FALLING_ACTION]",955,5025
2,expr_00020,S3[4:5]::entering || enables || S1[2:4]::check...,"[PRECONDITION, FALLING_ACTION]",955,5036
3,expr_00021,S3[11:12]::facing || motivates || S1[2:4]::che...,"[PRECONDITION, FALLING_ACTION]",955,4003
4,expr_00022,S3[15:16]::arrest || enables || S3[24:26]::che...,"[PRECONDITION, FALLING_ACTION]",955,5016
5,expr_00023,S3[17:18]::violating || enables || S3[15:16]::...,"[PRECONDITION, FALLING_ACTION]",955,5011
6,expr_00024,S3[17:18]::violating || enables || S3[24:26]::...,"[PRECONDITION, FALLING_ACTION]",955,5035
7,expr_00025,S3[24:26]::checked into || enables || S3[31:32...,"[PRECONDITION, FALLING_ACTION]",955,5027
8,expr_00026,S5[6:7]::comes || comes after || S3[24:26]::ch...,"[PRECONDITION, FALLING_ACTION]",955,4544
9,expr_00027,S5[6:7]::comes || comes after || S3[31:32]::begin,"[PRECONDITION, FALLING_ACTION]",955,4526


Layer 2 calls: 26
Mean Layer 2 user characters: 4688.4
Maximum Layer 2 user characters: 5036
Candidates per decision: [2]


## 7. Strict gold-relation trace

In [9]:
trace_df = pd.read_csv(RUN_DIR / "analysis/gold_relation_trace.csv")
display(trace_df)
display(
    trace_df.groupby("first_failure", dropna=False)
    .size()
    .reset_index(name="gold_relations")
    .sort_values("gold_relations", ascending=False)
)

,source_event_id,relation_id,target_event_id,source_keys,target_keys,first_failure
0,EVENT_2ea9a901cc230afcc371bcffee02a551,FALLING_ACTION,EVENT_401ce8fe35db27ca7b5f7cf83acc4084,"[""S3[39:40]::stay""]","[""S5[9:12]::rear - ended""]",layer01_relation_instance_missing
1,EVENT_2ea9a901cc230afcc371bcffee02a551,FALLING_ACTION,EVENT_7d29e127b36f34a94ceb0f13f9ea3a3b,"[""S3[39:40]::stay""]","[""S3[44:45]::conviction""]",layer01_relation_instance_missing
2,EVENT_2ea9a901cc230afcc371bcffee02a551,FALLING_ACTION,EVENT_a8934f381d6d835057ba8c537386ff9e,"[""S3[39:40]::stay""]","[""S5[27:28]::lied""]",layer01_relation_instance_missing
3,EVENT_2ea9a901cc230afcc371bcffee02a551,FALLING_ACTION,EVENT_f83b48c549abe22d26dc21624d8de397,"[""S3[39:40]::stay""]","[""S5[31:32]::telling""]",layer01_relation_instance_missing
4,EVENT_34cccabacad4dea93d3e762114dc05cd,FALLING_ACTION,EVENT_b79df11f8737f700691af4f8a7132190,"[""S1[2:4]::checks into""]","[""S3[4:5]::entering""]",layer01_relation_instance_missing
5,EVENT_34cccabacad4dea93d3e762114dc05cd,FALLING_ACTION,EVENT_c332a6b60af0495f34856f112dc68632,"[""S1[2:4]::checks into""]","[""S3[15:16]::arrest""]",layer01_relation_instance_missing
6,EVENT_34cccabacad4dea93d3e762114dc05cd,PRECONDITION,EVENT_2ea9a901cc230afcc371bcffee02a551,"[""S1[2:4]::checks into""]","[""S3[39:40]::stay""]",survived_to_layer05
7,EVENT_34cccabacad4dea93d3e762114dc05cd,PRECONDITION,EVENT_a6c700ffee90bb400461d1687e2287b9,"[""S1[2:4]::checks into""]","[""S5[2:3]::stay""]",layer01_target_event_missing
8,EVENT_434faad0c9ec71baa6a91df3c385299b,FALLING_ACTION,EVENT_70540222392ed30d611c5073a5b3204c,"[""S4[15:16]::violation""]","[""S4[21:22]::case""]",layer01_target_event_missing
9,EVENT_7d29e127b36f34a94ceb0f13f9ea3a3b,PRECONDITION,EVENT_a6c700ffee90bb400461d1687e2287b9,"[""S3[44:45]::conviction""]","[""S5[2:3]::stay""]",layer01_target_event_missing


,first_failure,gold_relations
0,layer01_relation_instance_missing,11
2,layer01_target_event_missing,4
1,layer01_source_event_missing,3
3,survived_to_layer05,2


## 8. Native event candidates, assertions and triples

In [10]:
states = {index: state for index, _, state in load_layer_states(RUN_DIR)}
layer3 = states.get(3)
layer4 = states.get(4)
layer5 = states.get(5)
layer6 = states.get(6)
layer11 = states.get(11)

if layer3:
    print("Layer 3 event candidates")
    display(pd.DataFrame([{
        "candidate_id": c.candidate_id,
        "canonical_label": c.canonical_label,
        "mentions": [m.text for m in c.mentions],
        "ontology_hints": c.ontology_hints,
    } for c in layer3.event_candidates or []]))

    print("Layer 3 relation candidates")
    display(pd.DataFrame([{
        "candidate_id": c.candidate_id,
        "canonical_label": c.canonical_label,
        "mentions": [m.text for m in c.mentions],
        "controlled_hints": [h for h in c.ontology_hints if str(h).lower().startswith("controlled_relation:")],
    } for c in layer3.relation_candidates or []]))
    assert all(c.mentions for c in layer3.relation_candidates or []), "Mention-free relation candidate detected."

if layer4:
    print("Layer 4 assertions")
    display(pd.DataFrame([{
        "source": x.source_candidate_label,
        "predicate": x.relation_label,
        "target": x.target_candidate_label,
        "confidence": x.confidence,
    } for x in layer4.candidate_relation_assertions or []]))

if layer5:
    print("Layer 5 triples")
    display(pd.DataFrame([{
        "subject": x.subject_label,
        "predicate": x.predicate_label,
        "object": x.object_label,
        "confidence": x.confidence,
    } for x in layer5.candidate_triples or []]))

print("Layer 6 ontology relation candidates:", len(layer6.ontology_relation_candidates or []) if layer6 else None)
print("Layer 11 completion candidates:", len(layer11.completion_candidates or []) if layer11 else None)

Layer 3 event candidates


,candidate_id,canonical_label,mentions,ontology_hints
0,cand_s_00000,S1[2:4]::checks into,[S1[2:4]::checks into],"[semantic_role:event_mention, candidate_family..."
1,cand_s_00001,S3[1:2]::skipping,[S3[1:2]::skipping],"[semantic_role:event_mention, candidate_family..."
2,cand_s_00002,S3[4:5]::entering,[S3[4:5]::entering],"[semantic_role:event_mention, candidate_family..."
3,cand_s_00003,S3[11:12]::facing,[S3[11:12]::facing],"[semantic_role:event_mention, candidate_family..."
4,cand_s_00004,S3[15:16]::arrest,[S3[15:16]::arrest],"[semantic_role:event_mention, candidate_family..."
5,cand_s_00005,S3[17:18]::violating,[S3[17:18]::violating],"[semantic_role:event_mention, candidate_family..."
6,cand_s_00006,S3[24:26]::checked into,[S3[24:26]::checked into],"[semantic_role:event_mention, candidate_family..."
7,cand_s_00007,S3[31:32]::begin,[S3[31:32]::begin],"[semantic_role:event_mention, candidate_family..."
8,cand_s_00008,S3[39:40]::stay,[S3[39:40]::stay],"[semantic_role:event_mention, candidate_family..."
9,cand_s_00009,S3[44:45]::conviction,[S3[44:45]::conviction],"[semantic_role:event_mention, candidate_family..."


Layer 3 relation candidates


,candidate_id,canonical_label,mentions,controlled_hints
0,cand_r_00000,PRECONDITION,[S1[2:4]::checks into || enables || S3[39:40]:...,[controlled_relation:PRECONDITION]
1,cand_r_00001,PRECONDITION,[S3[1:2]::skipping || enables || S1[2:4]::chec...,[controlled_relation:PRECONDITION]
2,cand_r_00002,PRECONDITION,[S3[4:5]::entering || enables || S1[2:4]::chec...,[controlled_relation:PRECONDITION]
3,cand_r_00003,PRECONDITION,[S3[11:12]::facing || motivates || S1[2:4]::ch...,[controlled_relation:PRECONDITION]
4,cand_r_00004,PRECONDITION,[S3[15:16]::arrest || enables || S3[24:26]::ch...,[controlled_relation:PRECONDITION]
5,cand_r_00005,PRECONDITION,[S3[17:18]::violating || enables || S3[15:16]:...,[controlled_relation:PRECONDITION]
6,cand_r_00006,PRECONDITION,[S3[17:18]::violating || enables || S3[24:26]:...,[controlled_relation:PRECONDITION]
7,cand_r_00007,PRECONDITION,[S3[24:26]::checked into || enables || S3[31:3...,[controlled_relation:PRECONDITION]
8,cand_r_00008,FALLING_ACTION,[S5[6:7]::comes || comes after || S3[24:26]::c...,[controlled_relation:FALLING_ACTION]
9,cand_r_00009,PRECONDITION,[S5[6:7]::comes || comes after || S3[31:32]::b...,[controlled_relation:PRECONDITION]


Layer 4 assertions


,source,predicate,target,confidence
0,S1[2:4]::checks into,PRECONDITION,S3[39:40]::stay,1.0
1,S3[1:2]::skipping,PRECONDITION,S1[2:4]::checks into,1.0
2,S3[4:5]::entering,PRECONDITION,S1[2:4]::checks into,1.0
3,S3[11:12]::facing,PRECONDITION,S1[2:4]::checks into,1.0
4,S3[15:16]::arrest,PRECONDITION,S3[24:26]::checked into,1.0
5,S3[17:18]::violating,PRECONDITION,S3[15:16]::arrest,1.0
6,S3[17:18]::violating,PRECONDITION,S3[24:26]::checked into,1.0
7,S3[24:26]::checked into,PRECONDITION,S3[31:32]::begin,1.0
8,S5[6:7]::comes,FALLING_ACTION,S3[24:26]::checked into,1.0
9,S5[6:7]::comes,PRECONDITION,S3[31:32]::begin,1.0


Layer 5 triples


,subject,predicate,object,confidence
0,S1[2:4]::checks into,PRECONDITION,S3[39:40]::stay,1.0
1,S3[1:2]::skipping,PRECONDITION,S1[2:4]::checks into,1.0
2,S3[4:5]::entering,PRECONDITION,S1[2:4]::checks into,1.0
3,S3[11:12]::facing,PRECONDITION,S1[2:4]::checks into,1.0
4,S3[15:16]::arrest,PRECONDITION,S3[24:26]::checked into,1.0
5,S3[17:18]::violating,PRECONDITION,S3[15:16]::arrest,1.0
6,S3[17:18]::violating,PRECONDITION,S3[24:26]::checked into,1.0
7,S3[24:26]::checked into,PRECONDITION,S3[31:32]::begin,1.0
8,S5[6:7]::comes,FALLING_ACTION,S3[24:26]::checked into,1.0
9,S5[6:7]::comes,PRECONDITION,S3[31:32]::begin,1.0


Layer 6 ontology relation candidates: 3
Layer 11 completion candidates: 0


## 9. Event projection audit

In [11]:
projection = pd.read_csv(RUN_DIR / "analysis/event_projection_audit.csv")
display(projection)

# Exact-span demonstration using source structure. Gold is consulted only here,
# after the pipeline has completed.
first_gold_key = next(iter(gold_event_index(gold)["keys_by_id"].values()))[0]
print(first_gold_key)
print(project_event_label(first_gold_key, gold))

,event_id,method,label,candidate_event_ids
0,EVENT_34cccabacad4dea93d3e762114dc05cd,exact_sentence_token_span,S1[2:4]::checks into,NaN
1,NaN,unmapped_or_ambiguous,S3[1:2]::skipping,[]
2,EVENT_b79df11f8737f700691af4f8a7132190,exact_sentence_token_span,S3[4:5]::entering,NaN
3,NaN,unmapped_or_ambiguous,S3[11:12]::facing,[]
4,EVENT_c332a6b60af0495f34856f112dc68632,exact_sentence_token_span,S3[15:16]::arrest,NaN
5,EVENT_9c07ef70f303f23dd82b32fb28a61ecb,exact_sentence_token_span,S3[17:18]::violating,NaN
6,EVENT_d7aec1e4ff20c3264f9fb7686355eb8a,exact_sentence_token_span,S3[24:26]::checked into,NaN
7,NaN,unmapped_or_ambiguous,S3[31:32]::begin,[]
8,EVENT_2ea9a901cc230afcc371bcffee02a551,exact_sentence_token_span,S3[39:40]::stay,NaN
9,EVENT_7d29e127b36f34a94ceb0f13f9ea3a3b,exact_sentence_token_span,S3[44:45]::conviction,NaN


S4[15:16]::violation
{'event_id': 'EVENT_434faad0c9ec71baa6a91df3c385299b', 'method': 'exact_sentence_token_span', 'label': 'S4[15:16]::violation'}


## 10. Speed, concurrency and errors

In [12]:
def optional_jsonl(path: Path) -> pd.DataFrame:
    return pd.DataFrame(read_jsonl(path)) if path.is_file() else pd.DataFrame()

calls = optional_jsonl(RUN_DIR / "run_logs/llm_calls.jsonl")
errors = optional_jsonl(RUN_DIR / "run_logs/llm_errors.jsonl")
parse_errors = optional_jsonl(RUN_DIR / "run_logs/llm_parse_errors.jsonl")
retrieval = optional_jsonl(RUN_DIR / "run_logs/ontology_retrieval.jsonl")

if not calls.empty:
    display(calls)
    display(calls.groupby("layer_tag").agg(
        calls=("call_index", "count"),
        total_recorded_seconds=("elapsed_seconds", "sum"),
        maximum_call_seconds=("elapsed_seconds", "max"),
        mean_system_chars=("system_chars", "mean"),
        mean_user_chars=("user_chars", "mean"),
        mean_response_chars=("response_chars", "mean"),
    ).reset_index())

print("Backend/API errors:", len(errors))
if not errors.empty: display(errors)
print("JSON parse errors:", len(parse_errors))
if not parse_errors.empty: display(parse_errors)
if not retrieval.empty:
    display(retrieval.groupby("layer_name").size().reset_index(name="retrieval_calls"))

manifest = read_json(RUN_DIR / "run_manifest.json")
print("Wall-clock pipeline seconds:", manifest.get("elapsed_seconds"))

,call_index,layer_tag,model,temperature,message_count,system_chars,user_chars,max_tokens,request_timeout,started_at,status,elapsed_seconds,response_chars,response_path,json_parse_ok,parsed_type,parse_error
0,3,layer01_event_instances,openai/gpt-oss-20b,0.0,2,1264,2756,8192,180,2026-07-31 01:38:40,ok,3.764,350,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
1,2,layer01_event_instances,openai/gpt-oss-20b,0.0,2,1264,3097,8192,180,2026-07-31 01:38:40,ok,6.145,1101,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
2,1,layer01_event_instances,openai/gpt-oss-20b,0.0,2,1264,2537,8192,180,2026-07-31 01:38:40,ok,21.209,199,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
3,4,layer01_event_instances,openai/gpt-oss-20b,0.0,2,1264,2907,8192,180,2026-07-31 01:38:40,ok,41.486,804,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
4,5,layer01_event_instances,openai/gpt-oss-20b,0.0,2,995,9621,8192,180,2026-07-31 01:39:21,ok,68.561,208,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
5,6,layer01_event_instances,openai/gpt-oss-20b,0.0,2,1064,9621,8192,180,2026-07-31 01:40:30,ok,40.274,420,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
6,8,layer01_event_instances,openai/gpt-oss-20b,0.0,2,1336,8995,8192,180,2026-07-31 01:41:10,ok,3.202,797,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
7,11,layer01_event_instances,openai/gpt-oss-20b,0.0,2,1336,8977,8192,180,2026-07-31 01:41:10,ok,3.702,515,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
8,9,layer01_event_instances,openai/gpt-oss-20b,0.0,2,1336,8995,8192,180,2026-07-31 01:41:10,ok,5.024,629,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
9,10,layer01_event_instances,openai/gpt-oss-20b,0.0,2,1336,8995,8192,180,2026-07-31 01:41:10,ok,9.768,2703,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None


,layer_tag,calls,total_recorded_seconds,maximum_call_seconds,mean_system_chars,mean_user_chars,mean_response_chars
0,layer01_event_instances,11,223.359,68.561,1254.090909,6863.272727,773.818182
1,layer02_event_relation_linking,26,175.182,30.989,955.000000,4688.384615,186.230769


Backend/API errors: 0
JSON parse errors: 0


,layer_name,retrieval_calls
0,layer02_candidate_enrichment,26


Wall-clock pipeline seconds: 203.571


## Success checklist before the five-document batch

1. The sibling RAGTree OWL-Time file resolves.
2. Parallel sentence extraction plus two whole-document reviews materially improves exact event recall.
3. Repeated mentions, eventive nominals, states, phrasal verbs, and hyphenated triggers retain exact indexed spans.
4. Layer 1B covers all validated sources in bounded batches and creates only closed-inventory pairs.
5. Layer 2 selects only `PRECONDITION`, `FALLING_ACTION`, or `found=false`; Layer 4 preserves direction.
6. Gold remains unavailable until post-Layer-12 evaluation. Freeze v1.3 before smoke-5 tuning.
